# 🇹🇳 Tunisian Dialect TTS — Fine-Tuning Notebook
**Model**: XTTS v2 (Coqui) fine-tuned on TunArTTS corpus  
**Language**: Tunisian Arabic (Darija)  


---
## Architecture Overview
```
TunArTTS Dataset (3h audio) → Preprocessing → XTTS v2 Fine-tune (GPT encoder) → Tunisian TTS
```
All env issues are pre-fixed: Python 3.11 venv, matplotlib backend, PyTorch 2.6 weights_only patch.

---
## CELL 1 — Install Python 3.11 + All Dependencies
**What it does**: Sets up an isolated Python 3.11 venv because TTS==0.22.0 requires Python < 3.12, but Colab now ships Python 3.12. All packages are installed in the venv.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 1 — Environment Setup
# Estimated time: 5–8 minutes
# ─────────────────────────────────────────────────────────────────────────────

import subprocess, sys

# Step 1: Install Python 3.11 system-wide
print("[1/5] Installing Python 3.11...")
!sudo apt-get update -q
!sudo apt-get install -y python3.11 python3.11-venv python3.11-dev -q
!apt-get install -y ffmpeg -q

# Step 2: Create isolated venv
print("[2/5] Creating Python 3.11 venv...")
!python3.11 -m venv /content/venv311

# Step 3: Upgrade pip inside venv
print("[3/5] Upgrading pip...")
!/content/venv311/bin/pip install --upgrade pip setuptools wheel -q

# Step 4: Install PyTorch first (specific version compatible with TTS 0.22.0)
print("[4/5] Installing PyTorch 2.1 (compatible with TTS 0.22.0)...")
!/content/venv311/bin/pip install torch==2.1.0 torchaudio==2.1.0 --index-url https://download.pytorch.org/whl/cu118 -q

# Step 5: Install TTS and supporting libraries
print("[5/5] Installing TTS and dependencies...")
!/content/venv311/bin/pip install TTS==0.22.0 -q
!/content/venv311/bin/pip install pydub ffmpeg-python -q
!/content/venv311/bin/pip install huggingface_hub datasets -q
!/content/venv311/bin/pip install matplotlib==3.7.5 -q  # pin to avoid backend issues
!/content/venv311/bin/pip install jiwer openai-whisper -q  # for evaluation
# Fix transformers compatibility: pin to 4.33.0 which is compatible with XTTS v2
!/content/venv311/bin/pip install transformers==4.33.0 tokenizers==0.13.3 -q

# Verify everything
print("\n" + "="*50)
print("VERIFICATION")
print("="*50)

# Write verification script to a file and execute it
verification_script = """
import sys
import torch
import TTS
import transformers

print(f'Python: {sys.version}')
print(f'PyTorch: {torch.__version__}')
print(f'Transformers: {transformers.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
print(f'TTS: {TTS.__version__}')
print("All good ✅")
"""

with open("/content/verify_env.py", "w") as f:
    f.write(verification_script)

!/content/venv311/bin/python /content/verify_env.py

[1/5] Installing Python 3.11...
Hit:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Hit:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease
Get:4 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Hit:5 https://r2u.stat.illinois.edu/ubuntu jammy InRelease
Hit:6 http://archive.ubuntu.com/ubuntu jammy InRelease
Hit:7 http://archive.ubuntu.com/ubuntu jammy-updates InRelease
Hit:8 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease
Hit:10 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:11 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu jammy InRelease
Get:12 http://security.ubuntu.com/ubuntu jammy-security/universe amd64 Packages [1,311 kB]
Fetched 1,444 kB in 2s (812 kB/s)
Reading package lists...
W: Skipping acquire of configure

---
## CELL 2 — Download TunArTTS Dataset
**What it does**: Downloads the elyadata/TunArTTS corpus (~1GB) — 3h+ of Tunisian speech with diacritized transcripts.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 2 — Download TunArTTS Dataset
# Estimated time: 5–15 minutes (depends on Colab bandwidth)
# ─────────────────────────────────────────────────────────────────────────────

from huggingface_hub import snapshot_download
import os

print("Downloading TunArTTS from HuggingFace...")
print("(No HF token needed — this is a public dataset)\n")

snapshot_download(
    repo_id="elyadata/TunArTTS",
    repo_type="dataset",
    local_dir="/content/TunArTTS",
    ignore_patterns=["*.metadata", ".gitattributes"]
)

print("\n✅ Download complete. Dataset structure:")
print("="*50)

for root, dirs, files in os.walk("/content/TunArTTS"):
    dirs[:] = [d for d in dirs if d not in ['__pycache__', '.cache']]
    level = root.replace("/content/TunArTTS", "").count(os.sep)
    if level < 3:
        indent = "  " * level
        print(f"{indent}{os.path.basename(root)}/")
        for f in files[:5]:
            print(f"{indent}  {f}")
        if len(files) > 5:
            print(f"{indent}  ... and {len(files)-5} more files")

# Count WAV files
import glob
wav_files = [f for f in glob.glob("/content/TunArTTS/dataset/wav/*")
             if f.endswith('.wav') and not f.endswith('.metadata')]
print(f"\n📢 Total WAV files found: {len(wav_files)}")

(No HF token needed — this is a public dataset)



/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Fetching ... files: 0it [00:00, ?it/s]


✅ Download complete. Dataset structure:
TunArTTS/
  README.md
  dataset/
    train.tsv
    wav/
      21109.wav
      15001.wav
      18187.wav
      19099.wav
      20278.wav
      ... and 1490 more files

📢 Total WAV files found: 1495


---
## CELL 3 — Inspect Metadata Structure
**What it does**: Examines the TSV file to understand column layout before building XTTS-format metadata. Critical — don't skip this.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 3 — Inspect Metadata (always run before building metadata.csv)
# ─────────────────────────────────────────────────────────────────────────────

import pandas as pd

TSV_PATH = "/content/TunArTTS/dataset/train.tsv"

# Load with header to see column names
df = pd.read_csv(TSV_PATH, sep="\t", nrows=10)

print("Column names:", df.columns.tolist())
print(f"Shape: {df.shape}")
print("\nFirst 5 rows:")
print(df.head())

# Show which columns contain Arabic text
print("\n" + "="*60)
print("COLUMN CONTENT PREVIEW")
print("="*60)
for col in df.columns:
    sample = str(df[col].iloc[0])[:80]
    print(f"  [{col}]: {sample}")

Column names: ['id', 'audio', 'sample_rate', 'speaker', 'duration', 'tgt_text_without_diacritization', 'tgt_text']
Shape: (10, 7)

First 5 rows:
      id      audio  sample_rate speaker   duration  \
0  21117  21117.wav        44100     SP1  10.588481   
1  18245  18245.wav        44100     SP1   9.370272   
2  21338  21338.wav        44100     SP1   9.526304   
3  19030  19030.wav        44100     SP1   8.362540   
4  19474  19474.wav        44100     SP1  10.560181   

                     tgt_text_without_diacritization  \
0  مشاعر قوية مشاعر قوية مشاعر قوية يكنلو مشاعر ق...   
1  مناعة مناعة مناعة عندو مناعة قوية ضد المرض عند...   
2  نبرة الصوت نبرة الصوت نبرة الصوت نبرة الصوت مت...   
3  عزيمة عزيمة عزيمة عندو عزيمة قوية عندو عزيمة قوية   
4  مشى في فاكونس مشى في فاكونس مشى في فاكونس مشى ...   

                                            tgt_text  
0  مَشَاعِرْ قْوِيَّة مَشَاعِرْ قْوِيَّة مَشَاعِر...  
1  مَنَاعَة مَنَاعَة مَنَاعَة عَنْدُو مَنَاعَة قْ...  
2  نَبْرِةْ الصُّوتْ ن

---
## CELL 4 — Preprocess Audio
**What it does**: Resamples all WAVs to 22050 Hz mono, splits on silence into 1.5–14s segments, and saves them to `/content/dataset/wavs/`. This is required by XTTS.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 4 — Audio Preprocessing
# Estimated time: 10–30 minutes depending on dataset size
# ─────────────────────────────────────────────────────────────────────────────

from pydub import AudioSegment
from pydub.silence import split_on_silence
import os, glob

RAW_DIR = "/content/TunArTTS/dataset/wav"
OUT_DIR = "/content/dataset/wavs"
os.makedirs(OUT_DIR, exist_ok=True)

# Safe glob — exclude .metadata sidecar files
wav_files = [
    f for f in glob.glob(f"{RAW_DIR}/*")
    if f.endswith('.wav') and not f.endswith('.metadata')
]

print(f"Found {len(wav_files)} WAV files in {RAW_DIR}")
print("Starting preprocessing (resample → split on silence → filter by duration)...\n")

seg_id  = 0
errors  = 0
skipped_short = 0
skipped_long  = 0

for i, wav_path in enumerate(wav_files):
    try:
        audio = AudioSegment.from_file(wav_path)
        # Resample to 22050 Hz mono — XTTS requirement
        audio = audio.set_frame_rate(22050).set_channels(1)

        chunks = split_on_silence(
            audio,
            min_silence_len=400,  # 400ms silence = sentence boundary
            silence_thresh=-38,   # dBFS threshold for silence detection
            keep_silence=80       # keep 80ms of silence on each side
        )

        for chunk in chunks:
            dur = len(chunk)  # milliseconds
            if dur < 1500:
                skipped_short += 1
                continue
            if dur > 14000:
                skipped_long += 1
                continue
            fname = f"seg_{seg_id:05d}.wav"
            chunk.export(f"{OUT_DIR}/{fname}", format="wav")
            seg_id += 1

    except Exception as e:
        errors += 1
        print(f"  ⚠ Skipped {os.path.basename(wav_path)}: {e}")

    # Progress every 100 files
    if (i + 1) % 100 == 0:
        print(f"  Processed {i+1}/{len(wav_files)} files → {seg_id} segments so far...")

print(f"\n{'='*50}")
print(f"✅ Preprocessing complete")
print(f"   Segments created  : {seg_id}")
print(f"   Skipped (too short): {skipped_short}")
print(f"   Skipped (too long) : {skipped_long}")
print(f"   Errors             : {errors}")
print(f"   Output directory   : {OUT_DIR}")

/usr/local/lib/python3.12/dist-packages/pydub/utils.py:300: SyntaxWarning: invalid escape sequence '\('
  m = re.match('([su]([0-9]{1,2})p?) \(([0-9]{1,2}) bit\)$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:301: SyntaxWarning: invalid escape sequence '\('
  m2 = re.match('([su]([0-9]{1,2})p?)( \(default\))?$', token)
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:310: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(flt)p?( \(default\))?$', token):
/usr/local/lib/python3.12/dist-packages/pydub/utils.py:314: SyntaxWarning: invalid escape sequence '\('
  elif re.match('(dbl)p?( \(default\))?$', token):


Found 1495 WAV files in /content/TunArTTS/dataset/wav
Starting preprocessing (resample → split on silence → filter by duration)...

  Processed 100/1495 files → 103 segments so far...
  Processed 200/1495 files → 208 segments so far...
  Processed 300/1495 files → 311 segments so far...
  Processed 400/1495 files → 422 segments so far...
  Processed 500/1495 files → 528 segments so far...
  Processed 600/1495 files → 634 segments so far...
  Processed 700/1495 files → 739 segments so far...
  Processed 800/1495 files → 847 segments so far...
  Processed 900/1495 files → 954 segments so far...
  Processed 1000/1495 files → 1056 segments so far...
  Processed 1100/1495 files → 1160 segments so far...
  Processed 1200/1495 files → 1261 segments so far...
  Processed 1300/1495 files → 1363 segments so far...
  Processed 1400/1495 files → 1466 segments so far...

✅ Preprocessing complete
   Segments created  : 1565
   Skipped (too short): 35
   Skipped (too long) : 8
   Errors             :

---
## CELL 5 — Build metadata.csv (XTTS format)
**What it does**: Reads the TSV, maps audio filenames to diacritized Arabic text, writes `filename|text|speaker` format that XTTS trainer expects. Uses `tgt_text` (diacritized) for best pronunciation results.

In [ ]:
import pandas as pd
import os
import glob

# ─── Paths ───
TSV_PATH = "/content/TunArTTS/dataset/train.tsv"
SEG_DIR  = "/content/dataset/wavs"
OUT_DIR  = "/content/dataset"
os.makedirs(OUT_DIR, exist_ok=True)

# ─── Load Original Data ───
df = pd.read_csv(TSV_PATH, sep="\t")

# ─── Map Segments to Text ───
seg_files = sorted(glob.glob(f"{SEG_DIR}/*.wav"))
print(f"Found {len(seg_files)} processed segments in {SEG_DIR}")

meta_data = []
for i, file_path in enumerate(seg_files):
    if i >= len(df):
        break

    # LJSpeech formatter adds .wav automatically, so we strip it here
    filename = os.path.basename(file_path).replace(".wav", "")
    text = str(df.iloc[i]['tgt_text'])

    # Clean text
    text = text.replace("|", " ")
    if len(text) > 5:
        meta_data.append([filename, text, "tunisian_speaker"])

meta = pd.DataFrame(meta_data, columns=["filename", "text", "speaker"])

# ─── Save ───
META_PATH = f"{OUT_DIR}/metadata.csv"
meta.to_csv(META_PATH, sep="|", index=False, header=False)

print(f"\n✅ metadata.csv saved with {len(meta)} valid mappings (extensions stripped).")
print(f"Sample row: {meta.iloc[0].tolist()}")

Found 1565 processed segments in /content/dataset/wavs

✅ metadata.csv saved with 1493 valid mappings (extensions stripped).
Sample row: ['seg_00000', 'مَشَاعِرْ قْوِيَّة مَشَاعِرْ قْوِيَّة مَشَاعِرْ قْوِيَّة يْكِنّْلُو مَشَاعِرْ قْوِيَّة يْكِنّْلُو مَشَاعِرْ قْوِيَّة', 'tunisian_speaker']


---
## CELL 6 — Tunisian Text Normalizer
**What it does**: Pre-processing function that standardizes Tunisian Darija spelling variants, converts French loanwords to Arabic phonetics, and removes characters that confuse the XTTS tokenizer.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 6 — Tunisian Text Normalizer (use this before every inference call)
# ─────────────────────────────────────────────────────────────────────────────

import re

# Tunisian dialect normalization rules
TUNISIAN_REPLACEMENTS = {
    # Standardize common Darija spelling variants
    "شنهو": "شنو",
    "شنية": "شنو",
    "كيفاه": "كيفاش",
    "كيفاهو": "كيفاش",
    "مانيش": "ما نيش",
    "مهوش": "ما هوش",
    "فمّا": "فما",
    "هوني": "هنا",
    "بالله": "بالله",
    "نحوس": "نحوس",
    # French loanwords → phonetic Arabic
    "merci": "مرسي",
    "bonne journée": "بون جورني",
    "pizza": "بيزا",
    "classe": "كلاس",
    "voiture": "فويتور",
    "portable": "بورتابل",
    "téléphone": "تيليفون",
    "ordinateur": "أورديناتور",
    # Numbers → Arabic words (Tunisian dialect)
    "1": "واحد",
    "2": "زوز",
    "3": "ثلاثة",
    "4": "أربعة",
    "5": "خمسة",
    "6": "ستة",
    "7": "سبعة",
    "8": "ثمانية",
    "9": "تسعة",
    "10": "عشرة",
}

def normalize_tunisian(text: str) -> str:
    """Normalize Tunisian Darija text for XTTS input."""
    text = text.strip()
    # Apply all replacements
    for src, tgt in TUNISIAN_REPLACEMENTS.items():
        text = text.replace(src, tgt)
    # Remove punctuation that confuses the TTS tokenizer
    text = re.sub(r'[،,؟?!:;«»\.]+', ' ', text)
    # Remove any lone digits still remaining
    text = re.sub(r'\b\d+\b', '', text)
    # Collapse multiple spaces
    text = re.sub(r'\s+', ' ', text).strip()
    return text

# ── Test the normalizer ──
test_cases = [
    "شنية تحب؟ مانيش عارف، merci برشا!",
    "عندي 3 ولاد وكيفاه حالهم",
    "روح للclimat واشري pizza",
    "كيفاش حالك؟ نتمنى باهي",
]

print("Normalizer test:")
print("="*60)
for t in test_cases:
    norm = normalize_tunisian(t)
    print(f"  IN : {t}")
    print(f"  OUT: {norm}")
    print()

print("✅ Normalizer ready — use normalize_tunisian(text) before every inference call")

Normalizer test:
  IN : شنية تحب؟ مانيش عارف، merci برشا!
  OUT: شنو تحب ما نيش عارف مرسي برشا

  IN : عندي 3 ولاد وكيفاه حالهم
  OUT: عندي ثلاثة ولاد وكيفاش حالهم

  IN : روح للclimat واشري pizza
  OUT: روح للclimat واشري بيزا

  IN : كيفاش حالك؟ نتمنى باهي
  OUT: كيفاش حالك نتمنى باهي

✅ Normalizer ready — use normalize_tunisian(text) before every inference call


---
## CELL 7 — Download XTTS v2 Base Model
**What it does**: Downloads the Coqui XTTS v2 checkpoint (~2GB). This is the starting point for fine-tuning — it already knows Arabic phonemes.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 7 — Download XTTS v2 Base Model
# Estimated time: 5–10 minutes
# ─────────────────────────────────────────────────────────────────────────────

from huggingface_hub import snapshot_download
import os

print("Downloading Coqui XTTS v2 base model (~2GB)...")

snapshot_download(
    repo_id="coqui/XTTS-v2",
    local_dir="/content/XTTS-v2"
)

print("\n✅ XTTS v2 downloaded")
print("Files:", os.listdir("/content/XTTS-v2"))

# Verify key files exist
required = ["model.pth", "config.json", "vocab.json"]
for f in required:
    path = f"/content/XTTS-v2/{f}"
    exists = os.path.exists(path)
    size   = os.path.getsize(path) / 1e6 if exists else 0
    status = f"✅ ({size:.0f} MB)" if exists else "❌ MISSING"
    print(f"  {f}: {status}")

Fetching 18 files:   0%|          | 0/18 [00:00<?, ?it/s]


✅ XTTS v2 downloaded
Files: ['vocab.json', 'README.md', 'hash.md5', 'LICENSE.txt', 'mel_stats.pth', 'samples', 'speakers_xtts.pth', 'config.json', '.gitattributes', 'dvae.pth', '.cache', 'model.pth']
  model.pth: ✅ (1868 MB)
  config.json: ✅ (0 MB)
  vocab.json: ✅ (0 MB)


---
## CELL 8 — Baseline Test (BEFORE Fine-Tuning)
**What it does**: Generates a sample with the vanilla XTTS v2 model. Listen to this — it's your baseline. After training, you'll compare against it.

In [ ]:
# ─────────────────────────────────────────────────────────────────────────────
# CELL 8 — Baseline Test BEFORE Fine-Tuning
# Listen to this first — it's your 'before' reference
# ─────────────────────────────────────────────────────────────────────────────

import glob, os

# Auto-pick a clean reference WAV from the dataset
wav_files = [
    f for f in glob.glob("/content/TunArTTS/dataset/wav/*.wav")
    if not f.endswith('.metadata')
]
# Sort for reproducibility, pick a mid-range file (avoids edge cases)
wav_files.sort()
REFERENCE_WAV = wav_files[len(wav_files)//2]  # pick a middle file
print(f"Reference WAV: {REFERENCE_WAV}")

# Save reference globally for use in later cells
with open("/content/reference_wav_path.txt", "w") as f:
    f.write(REFERENCE_WAV)

# Write test script — all TTS code runs through venv311 to avoid Python 3.12 conflict
test_sentences = [
    "كيفاش حالك نتمنى باهي",
    "شنو تحب تاكل اليوم",
    "برشا وقت ما شفتكش كيفاش العيلة",
]

script = f'''
import os
os.environ["MPLBACKEND"] = "agg"  # prevent Colab inline backend crash
os.environ["COQUI_TOS_AGREED"] = "1" # Auto-accept Coqui TOS

import torch

# PyTorch 2.6 compatibility patch — weights_only default changed
_orig_load = torch.load
def _safe_load(*args, **kwargs):
    kwargs.setdefault("weights_only", False)
    return _orig_load(*args, **kwargs)
torch.load = _safe_load

import matplotlib
matplotlib.use("agg")

from TTS.api import TTS

tts = TTS("tts_models/multilingual/multi-dataset/xtts_v2", gpu=torch.cuda.is_available())

sentences = {test_sentences}
for i, text in enumerate(sentences):
    out_path = f"/content/baseline_{{i}}.wav"
    tts.tts_to_file(
        text=text,
        speaker_wav="{REFERENCE_WAV}",
        language="ar",
        file_path=out_path
    )
    print(f"Saved baseline {{i}}: {{out_path}}")

print("Baseline generation complete")
'''

with open("/content/run_baseline.py", "w", encoding="utf-8") as f:
    f.write(script)

!/content/venv311/bin/python /content/run_baseline.py

Reference WAV: /content/TunArTTS/dataset/wav/15897.wav
/content/venv311/lib/python3.11/site-packages/TTS/api.py:70: UserWarning: `gpu` will be deprecated. Please use `tts.to(device)` instead.
  warnings.warn("`gpu` will be deprecated. Please use `tts.to(device)` instead.")
 > tts_models/multilingual/multi-dataset/xtts_v2 is already downloaded.
 > Using model: xtts
 > Text splitted to sentences.
['كيفاش حالك نتمنى باهي']
 > Processing time: 5.392622470855713
 > Real-time factor: 1.5958144390483207
Saved baseline 0: /content/baseline_0.wav
 > Text splitted to sentences.
['شنو تحب تاكل اليوم']
 > Processing time: 1.8160960674285889
 > Real-time factor: 0.4559678252732782
Saved baseline 1: /content/baseline_1.wav
 > Text splitted to sentences.
['برشا وقت ما شفتكش كيفاش العيلة']
 > Processing time: 2.0788698196411133
 > Real-time factor: 0.45787797190233487
Saved baseline 2: /content/baseline_2.wav
Baseline generation complete


In [ ]:
# Play baseline results
from IPython.display import Audio, display
import glob

sentences = [
    "كيفاش حالك نتمنى باهي",
    "شنو تحب تاكل اليوم",
    "برشا وقت ما شفتكش كيفاش العيلة",
]

for i, f in enumerate(sorted(glob.glob("/content/baseline_*.wav"))):
    print(f"\n▶ [{i}] {sentences[i] if i < len(sentences) else ''}")
    display(Audio(f))


▶ [0] كيفاش حالك نتمنى باهي



▶ [1] شنو تحب تاكل اليوم



▶ [2] برشا وقت ما شفتكش كيفاش العيلة
